In [12]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

# Load CSV files
# comparison_df = pd.read_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/compareContribution.csv")
comparison_df = pd.read_csv('/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/sciqa-comparisonContribution-Label.csv')
hq_df = pd.read_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/sciqa-comparisonHQs.csv")

# Load Sentence-BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast, and good quality

print("Encoding comparision label...")
comparison_labels = [str(i) for i in comparison_df["comparisonLabel"].tolist()]
comparison_IRIs = [str(i) for i in comparison_df["comparisonIRI"].tolist()]

target_embedding = model.encode(comparison_labels, convert_to_tensor=True)

print("\nEncoding HQ questions and finding most similar comparison label...")
predicted_comparison_label = []
predicted_comparison_IRI = []
cosine_scores_list = []
correct_predictions = []

for i, hq in enumerate(hq_df["question_string"]):
    print(f"\nEncoding HQ question {i+1}/{len(hq_df)}")
    source_embedding = model.encode(hq, convert_to_tensor=True)
    # Compute cosine similarities
    cosine_scores = cosine_similarity([source_embedding.cpu().numpy()], target_embedding.cpu().numpy())[0]
    # Find index of best match
    # find the index of the top 3 highest scores
    top_3_indices = np.argsort(cosine_scores)[-3:][::-1]
    print(cosine_scores[top_3_indices])
    cosine_scores_list.append(cosine_scores[top_3_indices])
    top3_candidates = [comparison_df.iloc[idx]['comparisonLabel'] for idx in top_3_indices]
    top3_candidate_IRIs = [comparison_df.iloc[idx]['comparisonIRI'] for idx in top_3_indices]
    # find the candidate comparison 
    goldComparisonLabel = hq_df.iloc[i]['goldComparisonLabel']
    print(goldComparisonLabel)
    print(top3_candidates)
    if goldComparisonLabel in top3_candidates:
        correct_predictions.append(1)
        print("Correct prediction")
    else:
        correct_predictions.append(0)
    print(f"Top 3 candidates for HQ question '{hq}':")
    for candidate in top3_candidates:
        print(f"  Top match candidate: {candidate}")
    predicted_comparison_label.append(top3_candidates)
    predicted_comparison_IRI.append(top3_candidate_IRIs)

hq_df["predicted_comparison_label"] = predicted_comparison_label
hq_df["predicted_comparison_IRI"] = predicted_comparison_IRI
hq_df["cosine_scores"] = cosine_scores_list
hq_df["correct_predictions"] = correct_predictions
hq_df.to_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/hq_with_sciqa_compareContribution.csv", index=False)

Encoding comparision label...

Encoding HQ questions and finding most similar comparison label...

Encoding HQ question 1/60
[0.81868416 0.29212445 0.29212445]
Summarization before 2002
['Summarization before 2002', 'Semantic representations of scholarly communication', 'Semantic representations of scholarly communication']
Correct prediction
Top 3 candidates for HQ question 'What was the most common type of approach for summarization before 2002?':
  Top match candidate: Summarization before 2002
  Top match candidate: Semantic representations of scholarly communication
  Top match candidate: Semantic representations of scholarly communication

Encoding HQ question 2/60
[0.31580895 0.21628493 0.17624506]
Pulsed electric field (PEF) treatment for bioactive compounds extraction
['Pulsed electric field (PEF) treatment for bioactive compounds extraction', 'Toxins produced by phytopathogenic Pseudomonas species and pathovars', 'Examples of cyclodextrin-containing formulations for drug deli

In [26]:
from IPython.display import Markdown as md
from orkg import ORKG, Hosts
import warnings
warnings.filterwarnings('ignore')

orkg = ORKG(host=Hosts.PRODUCTION)
df = pd.read_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/hq_with_sciqa_compareContribution.csv")
df.head()

prompt_column = []
response_column = []

for i in range(len(df)):
    question_id = df.iloc[i]['question_id']
    question_string = df.iloc[i]['question_string']
    predicted_comparison_IRI = df.iloc[i]['predicted_comparison_IRI']
    goldComparionIRI = df.iloc[i]['goldComparisonIRI']
    prompts = []
    responses = []
    print(f"comparison IRIs: {predicted_comparison_IRI}")
    print(f"type of comparison IRIs: {type(predicted_comparison_IRI)}")
    # convert the string representation of list back to list
    predicted_comparison_IRI = eval(predicted_comparison_IRI)
    print(f"Converted comparison IRIs: {predicted_comparison_IRI}")
    print(f"type of converted comparison IRIs: {type(predicted_comparison_IRI)}")
    # iterate through the top 3 predicted comparison IRIs
    for i, comparison_id in enumerate(predicted_comparison_IRI):
        print(f"Generating prompt for question ID {question_id} with comparison ID {comparison_id}")
        try:
            comparison_df = orkg.contributions.compare_dataframe(comparison_id=comparison_id)
        # some IRIs may not be valid, so we need to handle the exception
        except Exception as e:
            print(f"Error occurred while fetching comparison dataframe: {e}")
            continue
        # convert the dataframe to a csv data
        csv_data = comparison_df.to_csv(index=False)
        prompt = f"""
        Given the following CSV data representing a comparison of scientific contributions, and a question.
        Analyze the data and provide insights to answer the question.
        CSV Data:
        {csv_data}
        Question: {question_string}.
        Answer:
        """
        print(prompt)
        prompts.append(prompt)
        # response = get_response_from_language_model(prompt)
        # responses.append(response)
    # store the prompts and responses in the original dataframe
    prompt_column.append(prompts)
    # response_column.append(responses)
    print(f"Processed question ID: {question_id}")
df['prompts'] = prompt_column
# df['responses'] = response_column
      

comparison IRIs: ['R6948', 'R8364', 'R8364']
type of comparison IRIs: <class 'str'>
Converted comparison IRIs: ['R6948', 'R8364', 'R8364']
type of converted comparison IRIs: <class 'list'>
Generating prompt for question ID HQ0013 with comparison ID R6948

        Given the following CSV data representing a comparison of scientific contributions, and a question.
        Analyze the data and provide insights to answer the question.
        CSV Data:
        Automatic Abstracting Research at Chemical Abstracts Service/<italic>Contribution 1</italic>,Automatic condensation of electronic publications by sentence selection/<italic>Contribution 1</italic>,Automated text summarization and the SUMMARIST system/<italic>Contribution 1</italic>,"Trainable, scalable summarization using robust NLP and machine learning/<italic>Contribution 1</italic>",Generating Natural Language Summaries from Multiple On-Line Sources/<italic>Contribution 1</italic>,Discourse Trees Are Good Indicators of Importance i

In [27]:
df.head()

,question_id,question_string,sparql_query,goldComparisonIRI,goldComparisonLabel,predicted_comparison_label,predicted_comparison_IRI,cosine_scores,correct_predictions,prompts
0,HQ0013,What was the most common type of approach for ...,SELECT ?approach ?approach_label\nWHERE {\n o...,R6948,Summarization before 2002,"['Summarization before 2002', 'Semantic repres...","['R6948', 'R8364', 'R8364']",[0.81868416 0.29212445 0.29212445],1,[\n Given the following CSV data repres...
1,HQ0061,Which vegetables are utilized for betanin extr...,"SELECT ?vegetables, ?vegetables_labels\nWHERE ...",R75363,Pulsed electric field (PEF) treatment for bioa...,['Pulsed electric field (PEF) treatment for bi...,"['R75363', 'R69027', 'R155621']",[0.31580895 0.21628493 0.17624506],1,[\n Given the following CSV data repres...
2,HQ0049,What is the average energy generation of all e...,SELECT (AVG(?elec_gen_value) AS ?average_elec_...,R153801,Comparison of Studies on Germany's Energy Supp...,"[""Comparison of Studies on Germany's Energy Su...","['R153801', 'R153801', 'R153801']",[0.45773014 0.45773014 0.45773014],1,[]
3,HQ0002,"What is the scope of ""Decentralised Authoring,...",SELECT ?scope \nWHERE {\n orkgr:R8364 orkgp:c...,R8364,Semantic representations of scholarly communic...,['Ontologies for describing scholarly articles...,"['R8342', 'R8342', 'R8342']",[0.3319203 0.3319203 0.3319203],0,[\n Given the following CSV data repres...
4,HQ0072,What is the most common drug in the studies?,"SELECT ?drug, ?drug_labels\nWHERE {\n orkgr:R...",R155621,Examples of cyclodextrin-containing formulatio...,['Examples of cyclodextrin-containing formulat...,"['R155621', 'R110361', 'R44980']",[0.28183842 0.27690575 0.27017918],1,[\n Given the following CSV data repres...


## Show the comparison Table

In [10]:
from IPython.display import Markdown as md
from orkg import ORKG, Hosts
import warnings
warnings.filterwarnings('ignore')

# comparison_id = input("Please enter the ID of the comparison you want to analyze: ")
comparison_id = "R6187"

orkg = ORKG(host=Hosts.PRODUCTION)
df = orkg.contributions.compare_dataframe(comparison_id=comparison_id)
# convert the dataframe to a csv data
csv_data = df.to_csv(index=False)
prompt = f"""
Given the following CSV data representing a comparison of scientific contributions, and a question.
Analyze the data and provide insights to answer the question.
CSV Data:
{csv_data}
Question: Provide a summary of the key differences and similarities among the contributions in the comparison.
Answer:
"""

print(prompt)
resource = orkg.resources.by_id(id=comparison_id)
md("## Title: {} ([View](https://orkg.org/comparison/{}))".format(resource.content["label"],comparison_id))


Given the following CSV data representing a comparison of scientific contributions, and a question.
Analyze the data and provide insights to answer the question.
CSV Data:
Author disambiguation using multi-aspect similarity indicators/<italic>Contribution 1</italic>,A Real-time Heuristic-based Unsupervised Method for Name Disambiguation in Digital Libraries/<italic>Contribution 1</italic>,A semi-supervised approach for author disambiguation in KDD CUP 2013/<italic>Contribution 1</italic>,Ethnicity Sensitive Author Disambiguation Using Semi-supervised Learning/<italic>Contribution 1</italic>,Disambiguating authors in citations on the web and authorship correlations/<italic>Contribution 1</italic>,Citation-based bootstrapping for large-scale author disambiguation/<italic>Contribution 1</italic>,Self-training author name disambiguation for information scarce scenarios/<italic>Contribution 1</italic>,A Unified Semi-supervised Framework for Author Disambiguation in Academic Social Network/

## Title: Semi-supervised author name disambiguation ([View](https://orkg.org/comparison/R6187))

In [28]:
from IPython.display import Markdown as md
from orkg import ORKG, Hosts
import warnings
warnings.filterwarnings('ignore')

# comparison_id = input("Please enter the ID of the comparison you want to analyze: ")
comparison_id = "R6187"
orkg = ORKG(host=Hosts.PRODUCTION)
df = orkg.contributions.compare_dataframe(comparison_id=comparison_id)

In [29]:
df.head()

,Author disambiguation using multi-aspect similarity indicators/<italic>Contribution 1</italic>,A Real-time Heuristic-based Unsupervised Method for Name Disambiguation in Digital Libraries/<italic>Contribution 1</italic>,A semi-supervised approach for author disambiguation in KDD CUP 2013/<italic>Contribution 1</italic>,Ethnicity Sensitive Author Disambiguation Using Semi-supervised Learning/<italic>Contribution 1</italic>,Disambiguating authors in citations on the web and authorship correlations/<italic>Contribution 1</italic>,Citation-based bootstrapping for large-scale author disambiguation/<italic>Contribution 1</italic>,Self-training author name disambiguation for information scarce scenarios/<italic>Contribution 1</italic>,A Unified Semi-supervised Framework for Author Disambiguation in Academic Social Network/<italic>Contribution 1</italic>,Robust hybrid name disambiguation framework for large databases/<italic>Contribution 1</italic>
Method,Community detection algorithm<break/>Logistic ...,Heuristic-based<break/>Unsupervised and Adaptive,Community detection algorithm<break/>Support V...,First gradient boosted tree applied on similar...,Web and authorship correlations,Self-citation clustering rules with other rules,First pure clusters of data are found and mode...,Similar authors share co-authors and have high...,Web page genre identification based graph re-c...
uses similarity,Tani Moto coefficient,Levenshtein,Cosine<break/>Levenshtein distance<break/>TF-I...,Cosine<break/>Jaro–Winkler<break/>TF-IDF,Cosine<break/>Modified sigmoid function<break/...,Cosine<break/>TF-IDF,Cosine<break/>Euclidean,Cosine<break/>TF-IDF<break/>Tanimoto,Multi-dimensional scaling
Uncertainty,T,F,T,F,F,F,F,F,Robust
dataset,Self-designed WoS,DBLP,Microsoft Academic Service,INSPIRE,DBLP,WoK Thomson Reuters,BDBComp<break/>DBLP<break/>SyGAR,Microsoft Academic Service,DBLP
Performance metric,F-measure<break/>Precision<break/>Recall,F1<break/>Precision<break/>Recall,F1<break/>Precision<break/>Recall,F1<break/>Pairwise Precision<break/>Recall,F1<break/>Pairwise Precision<break/>Recall,F1<break/>Pairwise Precision<break/>Recall,K metric<break/>Pairwise F1,F1,F1


In [4]:
row_labels = df.index.tolist()
column_labels = df.columns.tolist()
print(row_labels)
print(column_labels)
for row in df.index:
    for col in df.columns:
        print(row, col, df.loc[row, col])


['Method', 'uses similarity', 'Uncertainty', 'dataset', 'Performance metric', 'deals with', 'Evidence', 'Limitations']
['Author disambiguation using multi-aspect similarity indicators/<italic>Contribution 1</italic>', 'A Real-time Heuristic-based Unsupervised Method for Name Disambiguation in Digital Libraries/<italic>Contribution 1</italic>', 'A semi-supervised approach for author disambiguation in KDD CUP 2013/<italic>Contribution 1</italic>', 'Ethnicity Sensitive Author Disambiguation Using Semi-supervised Learning/<italic>Contribution 1</italic>', 'Disambiguating authors in citations on the web and authorship correlations/<italic>Contribution 1</italic>', 'Citation-based bootstrapping for large-scale author disambiguation/<italic>Contribution 1</italic>', 'Self-training author name disambiguation for information scarce scenarios/<italic>Contribution 1</italic>', 'A Unified Semi-supervised Framework for Author Disambiguation in Academic Social Network/<italic>Contribution 1</italic>

## Enbedding questions and candidate entities from the KG

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm
import os

# comparison_df = pd.read_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/compareContribution.csv")
entity_df = pd.read_csv('/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/orkg_entity_labels.csv')
hq_df = pd.read_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/sciqa-comparisonHQs.csv")
candidates = entity_df["label"].tolist()
# Load Sentence-BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast, and good quality

print("Encoding entity label...")
entity_labels = [str(i) for i in entity_df["label"].tolist()]

embeddings_path = "entity_embeddings.npy"
if os.path.exists(embeddings_path):
    print(f"Loading embeddings from cache: {embeddings_path}")
    target_embeddings = np.load(embeddings_path)
    # ensure shape matches
    if target_embeddings.shape[0] != len(candidates):
        print("Warning: cached embeddings length differs from candidates; recomputing.")
else:
    target_embeddings = model.encode(entity_labels, convert_to_tensor=True)
    np.save(embeddings_path, target_embeddings.cpu().numpy())


print("\nEncoding HQ questions and finding most similar entity label...")
results = []
cosine_scores_list = []
correct_predictions = []

for i, hq in enumerate(hq_df["question_string"]):
    print(f"\nEncoding HQ question {i+1}/{len(hq_df)}")
    source_embedding = model.encode(hq, convert_to_tensor=True)
    # Compute cosine similarities
    cosine_scores = cosine_similarity([source_embedding.cpu().numpy()], target_embedding.cpu().numpy())[0]
    # Find index of best match
    # find the index of the top 3 highest scores
    top_3_indices = np.argsort(cosine_scores)[-3:][::-1]
    print(cosine_scores[top_3_indices])
    cosine_scores_list.append(cosine_scores[top_3_indices])
    top3_candidates = [entity_df.iloc[idx]['label'] for idx in top_3_indices]
    goldComparisonLabel = hq_df.iloc[i]['goldComparisonLabel']
    print(goldComparisonLabel)
    print(top3_candidates)
    if goldComparisonLabel in top3_candidates:
        correct_predictions.append(1)
        print("Correct prediction")
    else:
        correct_predictions.append(0)
    print(f"Top 3 candidates for HQ question '{hq}':")
    for candidate in top3_candidates:
        print(f"  Top match candidate: {candidate}")
    results.append(top3_candidates)

hq_df["predicted_entity_label"] = results
hq_df["cosine_scores"] = cosine_scores_list
hq_df["correct_predictions"] = correct_predictions
hq_df.to_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/sciqa/project_data/hq_with_candidate_entity_labels.csv", index=False)

/Users/sherrypan/miniconda3/envs/graphrag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoding entity label...

Encoding HQ questions and finding most similar entity label...

Encoding HQ question 1/60
[0.8186841  0.75431174 0.7377292 ]
Summarization before 2002
['Summarization before 2002', 'Summarization techniques', 'Recent summarization efforts']
Correct prediction
Top 3 candidates for HQ question 'What was the most common type of approach for summarization before 2002?':
  Top match candidate: Summarization before 2002
  Top match candidate: Summarization techniques
  Top match candidate: Recent summarization efforts

Encoding HQ question 2/60
[0.59633607 0.52660865 0.52005404]
Pulsed electric field (PEF) treatment for bioactive compounds extraction
['Betanin', 'Nutrient extraction', 'Vegetables and their products']
Top 3 candidates for HQ question 'Which vegetables are utilized for betanin extraction?':
  Top match candidate: Betanin
  Top match candidate: Nutrient extraction
  Top match candidate: Vegetables and their products

Encoding HQ question 3/60
[0.727267

In [1]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL

# Define the ontology schema content
ontology_schema = """
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix bf: <http://id.loc.gov/ontologies/bibframe/> .
@prefix bibo: <http://purl.org/ontology/bibo/> .
@prefix bibtex: <http://purl.org/net/nknouf/ns/bibtex#> .
@prefix cito: <http://purl.org/spar/cito/> .
@prefix datacite: <http://purl.org/spar/datacite/> .
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix litre: <http://purl.org/spar/literal/> .
@prefix locid: <http://id.loc.gov/vocabulary/identifiers/> .
@prefix locrel: <http://id.loc.gov/vocabulary/relators/> .
@prefix schema: <https://schema.org/> .
@prefix wd: <http://www.wikidata.org/entity/> .
@prefix wdt: <http://www.wikidata.org/prop/direct/> .
@prefix dblp: <https://dblp.org/rdf/schema#> .

<https://dblp.org/rdf/schema>
    dct:abstract "The dblp RDF schema is an ontology that models the semantic contents of the dblp computer science bibliography."@en ;
    dct:creator <https://dblp.org> ;
    dct:description "This ontology describes the RDF representation of the contents of the dblp computer science bibliography."@en ;
    dct:title "dblp RDF schema" ;
    a owl:Ontology ;
    owl:priorVersion <https://dblp.org/rdf/schema-2023-10-17> ;
    owl:versionIRI <https://dblp.org/rdf/schema-2024-06-14> ;
    owl:versionInfo "Fri 14 Jun 2024" .

dblp:AmbiguousCreator
    a rdfs:Class, owl:Class ;
    rdfs:comment "Not an actual creator, but an ambiguous proxy for an unknown number of unrelated actual creators of the same name. Associated publications do not have their true creators determined yet."@en ;
    rdfs:label "AmbiguousCreator"@en ;
    rdfs:subClassOf wd:Q48522, dblp:Creator .

dblp:Article
    a rdfs:Class, owl:Class ;
    rdfs:comment "A journal article."@en ;
    rdfs:label "Article"@en ;
    rdfs:subClassOf bibo:AcademicArticle, wd:Q13442814, dblp:Publication, schema:Article ;
    owl:equivalentClass dbo:Article .

dblp:AuthorSignature
    a rdfs:Class, owl:Class ;
    rdfs:comment "The information that links a publication to an author."@en ;
    rdfs:label "AuthorSignature"@en ;
    rdfs:subClassOf dblp:Signature .

dblp:Book
    a rdfs:Class, owl:Class ;
    rdfs:comment "A book or a thesis."@en ;
    rdfs:label "Book"@en ;
    rdfs:subClassOf dblp:Publication ;
    owl:equivalentClass dbo:Book, bibo:Book, wd:Q571, schema:Book .

dblp:Conference
    a rdfs:Class, owl:Class ;
    rdfs:comment "A conference or workshop series."@en ;
    rdfs:label "Conference"@en ;
    rdfs:subClassOf dblp:Stream ;
    owl:equivalentClass wd:Q47258130 .

dblp:Creator
    a rdfs:Class, owl:Class ;
    rdfs:comment "A creator of a publication."@en ;
    rdfs:label "Creator"@en ;
    rdfs:subClassOf dbo:Agent, bf:Agent, dct:Agent, wd:Q24229398, foaf:Agent, dblp:Entity .

dblp:Data
    a rdfs:Class, owl:Class ;
    rdfs:comment "Research data or artifacts."@en ;
    rdfs:label "Data"@en ;
    rdfs:subClassOf dblp:Publication ;
    owl:equivalentClass wd:Q17051824, schema:Dataset .

dblp:EditorSignature
    a rdfs:Class, owl:Class ;
    rdfs:comment "The information that links a publication to an editor."@en ;
    rdfs:label "EditorSignature"@en ;
    rdfs:subClassOf dblp:Signature .

dblp:Editorship
    a rdfs:Class, owl:Class ;
    rdfs:comment "An edited publication."@en ;
    rdfs:label "Editorship"@en ;
    rdfs:subClassOf dblp:Publication .

dblp:Entity
    a rdfs:Class, owl:Class ;
    rdfs:comment "An abstract, identifiable entity in dblp."@en ;
    rdfs:label "Entity"@en ;
    rdfs:subClassOf owl:Thing, wd:Q35120, schema:Thing .

dblp:Group
    a rdfs:Class, owl:Class ;
    rdfs:comment "A creator alias used by a group or consortium of persons."@en ;
    rdfs:label "Group"@en ;
    rdfs:subClassOf dblp:Creator ;
    owl:equivalentClass dbo:Organisation, bf:Organisation, wd:Q43229, foaf:Group, schema:Organization .

dblp:Incollection
    a rdfs:Class, owl:Class ;
    rdfs:comment "A part/chapter in a book or a collection."@en ;
    rdfs:label "Incollection"@en ;
    rdfs:subClassOf bibo:AcademicArticle, wd:Q13442814, dblp:Publication, schema:Chapter .

dblp:Informal
    a rdfs:Class, owl:Class ;
    rdfs:comment "An informal or other publication."@en ;
    rdfs:label "Informal"@en ;
    rdfs:subClassOf dblp:Publication ;
    owl:equivalentClass wd:Q1148359 .

dblp:Inproceedings
    a rdfs:Class, owl:Class ;
    rdfs:comment "A conference or workshop paper."@en ;
    rdfs:label "Inproceedings"@en ;
    rdfs:subClassOf bibo:AcademicArticle, wd:Q13442814, dblp:Publication, schema:Chapter .

dblp:Journal
    a rdfs:Class, owl:Class ;
    rdfs:comment "A periodically published journal."@en ;
    rdfs:label "Journal"@en ;
    rdfs:subClassOf dblp:Stream, schema:Periodical ;
    owl:equivalentClass wd:Q5633421 .

dblp:Person
    a rdfs:Class, owl:Class ;
    rdfs:comment "An actual person, who is a creator of a publication."@en ;
    rdfs:label "Person"@en ;
    rdfs:subClassOf dblp:Creator ;
    owl:equivalentClass dbo:Person, bf:Person, wd:Q5, foaf:Person, schema:Person .

dblp:Publication
    a rdfs:Class, owl:Class ;
    rdfs:comment "A publication."@en ;
    rdfs:label "Publication"@en ;
    rdfs:subClassOf dbo:WrittenWork, bf:Work, dct:BibliographicResource, bibo:Document, foaf:Document, dblp:Entity, schema:CreativeWork ;
    owl:equivalentClass wd:Q591041 .

dblp:Reference
    a rdfs:Class, owl:Class ;
    rdfs:comment "A reference work entry."@en ;
    rdfs:label "Reference"@en ;
    rdfs:subClassOf dblp:Publication ;
    owl:equivalentClass bibo:ReferenceSource, wd:Q10389811 .

dblp:Repository
    a rdfs:Class, owl:Class ;
    rdfs:comment "A source of data and/or artifact publications."@en ;
    rdfs:label "Repository"@en ;
    rdfs:subClassOf dblp:Stream ;
    owl:equivalentClass wd:Q5227240 .

dblp:Series
    a rdfs:Class, owl:Class ;
    rdfs:comment "A published series of volumes."@en ;
    rdfs:label "Series"@en ;
    rdfs:subClassOf dblp:Stream ;
    owl:equivalentClass wd:Q2217301 .

dblp:Signature
    a rdfs:Class, owl:Class ;
    rdfs:comment "The information that links a publication to a creator."@en ;
    rdfs:label "Signature"@en .

dblp:Stream
    a rdfs:Class, owl:Class ;
    rdfs:comment "A publication stream, i.e., a venue or source for publications."@en ;
    rdfs:label "Stream"@en ;
    rdfs:subClassOf dblp:Entity, schema:CreativeWorkSeries .

dblp:VersionRelation
    a rdfs:Class, owl:Class ;
    rdfs:comment "The information that links an (instanced) publication version to its general (concept) publication."@en ;
    rdfs:label "VersionRelation"@en .

dblp:Withdrawn
    a rdfs:Class, owl:Class ;
    rdfs:comment "A withdrawn publication item."@en ;
    rdfs:label "Withdrawn"@en ;
    rdfs:subClassOf dblp:Publication ;
    owl:equivalentClass wd:Q45182324 .

dblp:affiliation
    a rdf:Property ;
    rdfs:comment "A (past or present) affiliation of the creator. (Remark: This property currently just gives literal xsd:string values until institutions are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "affiliation"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dblp:creatorNote ;
    owl:equivalentProperty dbo:affiliation, wd:P1416, schema:affiliation .

dblp:archivedWebpage
    a rdf:Property ;
    rdfs:comment "The URL of an archived web page about this item, which may no longer be available in the web."@en ;
    rdfs:domain dblp:Entity ;
    rdfs:label "archivedWebpage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:webpage .

dblp:authorOf
    a rdf:Property ;
    rdfs:comment "The creator is the author of the publication."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "authorOf"@en ;
    rdfs:range dblp:Publication ;
    rdfs:subPropertyOf dblp:creatorOf ;
    owl:inverseOf dblp:authoredBy .

dblp:authoredBy
    a rdf:Property ;
    rdfs:comment "The publication is authored by the creator."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "authoredBy"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf dblp:createdBy ;
    owl:equivalentProperty dbo:author, locrel:aut, wd:P50, schema:author ;
    owl:inverseOf dblp:authorOf .

dblp:awardWebpage
    a rdf:Property ;
    rdfs:comment "The URL of a web page about an award received by this creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "awardwebpage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:webpage .

dblp:bibtexType
    a rdf:Property ;
    rdfs:comment "The bibtex type of the publication, e.g. book, inproceedings, etc."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "bibtexType"@en ;
    rdfs:range bibtex:Entry .

dblp:coAuthorWith
    a rdf:Property, owl:SymmetricProperty ;
    rdfs:comment "The creator is co-author with the other creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "coAuthorWith"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf dblp:coCreatorWith .

dblp:coCreatorWith
    a rdf:Property, owl:SymmetricProperty ;
    rdfs:comment "The creator is co-creator with the other creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "coCreatorWith"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf foaf:knows .

dblp:coEditorWith
    a rdf:Property, owl:SymmetricProperty ;
    rdfs:comment "The creator is co-editor with the other creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "coEditorWith"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf dblp:coCreatorWith .

dblp:createdBy
    a rdf:Property ;
    rdfs:comment "The publication is created by the creator."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "createdBy"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf dc:creator, foaf:maker ;
    owl:equivalentProperty locrel:cre, dct:creator, wd:P170, schema:creator ;
    owl:inverseOf dblp:creatorOf .

dblp:creatorName
    a rdf:Property ;
    rdfs:comment "The full name of the creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "creatorname"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P2561, foaf:name, schema:name .

dblp:creatorNote
    a rdf:Property ;
    rdfs:comment "An additional note about the creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "creatorNote"@en ;
    rdfs:range xsd:string .

dblp:creatorOf
    a rdf:Property ;
    rdfs:comment "The creator of the publication."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "creatorOf"@en ;
    rdfs:range dblp:Publication ;
    rdfs:subPropertyOf foaf:made ;
    owl:inverseOf dblp:createdBy .

dblp:documentPage
    a rdf:Property ;
    rdfs:comment "The URL of the electronic edition of the publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "documentPage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:webpage ;
    owl:equivalentProperty bf:electronicLocator, bibo:uri .

dblp:doi
    a rdf:Property ;
    rdfs:comment "A Digital Object Identifier."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "doi"@en ;
    rdfs:range xsd:anyUri ;
    rdfs:subPropertyOf dblp:identifier ;
    owl:equivalentProperty locid:doi, datacite:doi .

dblp:editedBy
    a rdf:Property ;
    rdfs:comment "The publication is edited by the creator."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "editedBy"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf dblp:createdBy ;
    owl:equivalentProperty dbo:editor, locrel:edt, bibo:editor, wd:P98, schema:editor ;
    owl:inverseOf dblp:editorOf .

dblp:editorOf
    a rdf:Property ;
    rdfs:comment "The creator is the editor of the publication."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "editorOf"@en ;
    rdfs:range dblp:Publication ;
    rdfs:subPropertyOf dblp:creatorOf ;
    owl:inverseOf dblp:editedBy .

dblp:formerStreamTitle
    a rdf:Property ;
    rdfs:comment "A former title of the stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "formerStreamTitle"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dblp:streamTitle .

dblp:hasSignature
    a rdf:Property ;
    rdfs:comment "A signature that links this publication to an creator."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "hasSignature"@en ;
    rdfs:range dblp:Signature ;
    owl:inverseOf dblp:signaturePublication .

dblp:hasVersion
    a rdf:Property ;
    rdfs:comment "The publication has a different, more specific (instance) publication as its version."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "hasVersion"@en ;
    rdfs:range dblp:VersionRelation ;
    owl:inverseOf dblp:versionConcept .

dblp:homepage
    a rdf:Property ;
    rdfs:comment "The URL of an academic homepage of this creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "homepage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:webpage ;
    owl:equivalentProperty wd:P856, foaf:homepage .

dblp:homonymousCreator
    a rdf:Property, owl:SymmetricProperty ;
    rdfs:comment "This creator shares a homonymous name with the other creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "homonymousCreator"@en ;
    rdfs:range dblp:Creator .

dblp:identifier
    a rdf:Property ;
    rdfs:comment "An abstract identifier."@en ;
    rdfs:domain dblp:Entity ;
    rdfs:label "identifier"@en ;
    rdfs:range xsd:anyUri ;
    owl:equivalentProperty locid:id .

dblp:indexPage
    a rdf:Property ;
    rdfs:comment "The URL of the dblp stream index page for this stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "indexPage"@en ;
    rdfs:range foaf:Document .

dblp:isVersion
    a rdf:Property ;
    rdfs:comment "The publication is a version of another, more general (concept) publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "isVersion"@en ;
    rdfs:range dblp:VersionRelation ;
    owl:inverseOf dblp:versionInstance .

dblp:isVersionOf
    a rdf:Property, owl:TransitiveProperty ;
    rdfs:comment "The publication is a version of another, more general (concept) publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "isVersionOf"@en ;
    rdfs:range dblp:Publication ;
    owl:equivalentProperty wd:P747 .

dblp:isbn
    a rdf:Property ;
    rdfs:comment "An International Standard Book Number."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "isbn"@en ;
    rdfs:range xsd:anyUri ;
    rdfs:subPropertyOf dblp:identifier ;
    owl:equivalentProperty locid:isbn, datacite:isbn .

dblp:iso4
    a rdf:Property ;
    rdfs:comment "The stream's ISO4 abbreviation."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "iso4"@en ;
    rdfs:range xsd:string ;
    owl:equivalentProperty wd:P1160 .

dblp:issn
    a rdf:Property ;
    rdfs:comment "An International Standard Serial Number."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "issn"@en ;
    rdfs:range xsd:anyUri ;
    rdfs:subPropertyOf dblp:identifier ;
    owl:equivalentProperty locid:issn, datacite:issn .

dblp:listedOnTocPage
    a rdf:Property ;
    rdfs:comment "The url of the dblp table of contents page listing this publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "listedOnTocPage"@en ;
    rdfs:range foaf:Document .

dblp:monthOfPublication
    a rdf:Property ;
    rdfs:comment "The month the publication has been published."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "monthOfPublication"@en ;
    rdfs:range xsd:gMonth ;
    rdfs:subPropertyOf wd:P2922 .

dblp:numberOfCreators
    a rdf:Property ;
    rdfs:comment "The number of creators who created this publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "numberOfCreators"@en ;
    rdfs:range xsd:integer .

dblp:omid
    a rdf:Property ;
    rdfs:comment "An OpenCitations Meta Identifier."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "omid"@en ;
    rdfs:range xsd:anyUri ;
    rdfs:subPropertyOf dblp:identifier ;
    owl:equivalentProperty datacite:omid .

dblp:orcid
    a rdf:Property ;
    rdfs:comment "An Open Researcher and Contributor ID."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "orcid"@en ;
    rdfs:range xsd:anyUri ;
    rdfs:subPropertyOf dblp:identifier ;
    owl:equivalentProperty locid:orcid, datacite:orcid .

dblp:pagination
    a rdf:Property ;
    rdfs:comment "The page numbers where the publication can be found."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "pagination"@en ;
    rdfs:range xsd:string ;
    owl:equivalentProperty bibo:pages, wd:P304, schema:pagination .

dblp:possibleActualCreator
    a rdf:Property ;
    rdfs:comment "This ambiguous creator may be (or may be not) just a disambiguation proxy for the other creator. Further actual creator candidates are possible."@en ;
    rdfs:domain dblp:AmbiguousCreator ;
    rdfs:label "possibleActualCreator"@en ;
    rdfs:range dblp:Creator ;
    rdfs:subPropertyOf dblp:homonymousCreator ;
    owl:inverseOf dblp:proxyAmbiguousCreator .

dblp:predecessorStream
    a rdf:Property ;
    rdfs:comment "This stream is a predecessor of the other stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "predecessorStream"@en ;
    rdfs:range dblp:Stream ;
    rdfs:subPropertyOf dblp:relatedStream ;
    owl:inverseOf dblp:successorStream .

dblp:primaryAffiliation
    a rdf:Property ;
    rdfs:comment "The primary affiliation of the creator. (Remark: This property currently just gives literal xsd:string values until institutions are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "primaryAffiliation"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dblp:affiliation .

dblp:primaryCreatorName
    a rdf:Property ;
    rdfs:comment "The primary full name of the creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "primaryCreatorName"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dbo:commonName, dblp:creatorName .

dblp:primaryDocumentPage
    a rdf:Property ;
    rdfs:comment "The primary URL of the electronic edition of the publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "primaryDocumentPage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:documentPage .

dblp:primaryHomepage
    a rdf:Property ;
    rdfs:comment "The primary URL of an academic homepage of this creator."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "primaryHomepage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:homepage .

dblp:primaryStreamTitle
    a rdf:Property ;
    rdfs:comment "The primary title of the stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "primaryStreamTitle"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dbo:commonName, dblp:streamTitle .

dblp:proxyAmbiguousCreator
    a rdf:Property ;
    rdfs:comment "This creator (and any of her fellow homonymous creators) is also represented by the given ambiguous creator in cases where the authorship of a publication is undetermined."@en ;
    rdfs:domain dblp:Creator ;
    rdfs:label "proxyAmbiguousCreator"@en ;
    rdfs:range dblp:AmbiguousCreator ;
    rdfs:subPropertyOf dblp:homonymousCreator ;
    owl:inverseOf dblp:possibleActualCreator .

dblp:publicationNote
    a rdf:Property ;
    rdfs:comment "An additional note to the publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publicationNote"@en ;
    rdfs:range xsd:string .

dblp:publishedAsPartOf
    a rdf:Property, owl:TransitiveProperty ;
    rdfs:comment "The publication has been published as a part of the other publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedAsPartOf"@en ;
    rdfs:range dblp:Publication ;
    rdfs:subPropertyOf wd:P1433, schema:isPartOf .

dblp:publishedBy
    a rdf:Property ;
    rdfs:comment "The publisher of the publication. (Remark: This property currently just gives literal xsd:string values until publishers are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedBy"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dc:publisher ;
    owl:equivalentProperty dct:publisher, wd:P123, schema:publisher .

dblp:publishedIn
    a rdf:Property ;
    rdfs:comment "The name of the series, the journal, or the book in which the publication has been published. (Remark: This property currently just gives literal xsd:string values until journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedIn"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P1433, schema:isPartOf .

dblp:publishedInBook
    a rdf:Property ;
    rdfs:comment "The name of the book in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInBook"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dblp:publishedIn .

dblp:publishedInBookChapter
    a rdf:Property ;
    rdfs:comment "The chapter of the book in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInBookChapter"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P792, schema:position ;
    owl:equivalentProperty bibo:chapter .

dblp:publishedInJournal
    a rdf:Property ;
    rdfs:comment "The name of the journal in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInJournal"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dblp:publishedIn .

dblp:publishedInJournalVolume
    a rdf:Property ;
    rdfs:comment "The volume of the journal in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInJournalVolume"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P478, schema:volumeNumber ;
    owl:equivalentProperty bibo:volume .

dblp:publishedInJournalVolumeIssue
    a rdf:Property ;
    rdfs:comment "The issue of the journal in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInJournalIssue"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P433, schema:issueNumber ;
    owl:equivalentProperty bibo:issue .

dblp:publishedInSeries
    a rdf:Property ;
    rdfs:comment "The name of the series in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInSeries"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dblp:publishedIn .

dblp:publishedInSeriesVolume
    a rdf:Property ;
    rdfs:comment "The volume of the series in which the publication has been published. (Remark: This is currently an intermediate property that will be removed once journals and conference series are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInSeriesVolume"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P478, schema:volumeNumber ;
    owl:equivalentProperty bibo:volume .

dblp:publishedInStream
    a rdf:Property ;
    rdfs:comment "The conference series, the journal, or the repository in which the publication has been published."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishedInStream"@en ;
    rdfs:range dblp:Stream ;
    rdfs:subPropertyOf wd:P1433, schema:isPartOf .

dblp:publishersAddress
    a rdf:Property ;
    rdfs:comment "The address of the publisher. (Remark: This is currently an intermediate property that will be removed once publishers are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "publishersAddress"@en ;
    rdfs:range xsd:string .

dblp:relatedStream
    a rdf:Property, owl:SymmetricProperty ;
    rdfs:comment "This stream is related to the other stream in some unspecified way."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "relatedStream"@en ;
    rdfs:range dblp:Stream .

dblp:signatureCreator
    a rdf:Property ;
    rdfs:comment "A linked creator of the publication."@en ;
    rdfs:domain dblp:Signature ;
    rdfs:label "signatureCreator"@en ;
    rdfs:range dblp:Creator .

dblp:signatureDblpName
    a rdf:Property ;
    rdfs:comment "A dblp name (including any possible trailing homonym number) that links the publication to a creator."@en ;
    rdfs:domain dblp:Signature ;
    rdfs:label "signatureDblpName"@en ;
    rdfs:range xsd:string .

dblp:signatureOrcid
    a rdf:Property ;
    rdfs:comment "An ORCID that links the publication to a creator."@en ;
    rdfs:domain dblp:Signature ;
    rdfs:label "signatureOrcid"@en ;
    rdfs:range xsd:anyUri .

dblp:signatureOrdinal
    a rdf:Property ;
    rdfs:comment "The ordinal number of this signature for the publication, starting with 1."@en ;
    rdfs:domain dblp:Signature ;
    rdfs:label "signatureOrdinal"@en ;
    rdfs:range xsd:integer .

dblp:signaturePublication
    a rdf:Property ;
    rdfs:comment "The publication of this signature."@en ;
    rdfs:domain dblp:Signature ;
    rdfs:label "signaturePublication"@en ;
    rdfs:range dblp:Publication ;
    owl:inverseOf dblp:hasSignature .

dblp:streamTitle
    a rdf:Property ;
    rdfs:comment "A title of the stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "streamTitle"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf wd:P2561, foaf:name, schema:name .

dblp:subStream
    a rdf:Property ;
    rdfs:comment "This stream is (or was) a part of the other stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "subStream"@en ;
    rdfs:range dblp:Stream ;
    rdfs:subPropertyOf dblp:relatedStream ;
    owl:inverseOf dblp:superStream .

dblp:successorStream
    a rdf:Property ;
    rdfs:comment "This stream is a successor of the other stream."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "successorStream"@en ;
    rdfs:range dblp:Stream ;
    rdfs:subPropertyOf dblp:relatedStream ;
    owl:inverseOf dblp:predecessorStream .

dblp:superStream
    a rdf:Property ;
    rdfs:comment "This stream has (or had) the other stream as a part."@en ;
    rdfs:domain dblp:Stream ;
    rdfs:label "superStream"@en ;
    rdfs:range dblp:Stream ;
    rdfs:subPropertyOf dblp:relatedStream ;
    owl:inverseOf dblp:subStream .

dblp:thesisAcceptedBySchool
    a rdf:Property ;
    rdfs:comment "The school where the publication (typically a thesis) has been accepted. (Remark: This property currently just gives literal xsd:string values until institutions are modelled as proper entities.)"@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "thesisAcceptedBySchool"@en ;
    rdfs:range xsd:string .

dblp:title
    a rdf:Property ;
    rdfs:comment "The title of the publication."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "title"@en ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf dc:title, schema:name ;
    owl:equivalentProperty dct:title, wd:P1476 .

dblp:versionConcept
    a rdf:Property ;
    rdfs:comment "The linked general (concept) publication."@en ;
    rdfs:domain dblp:VersionRelation ;
    rdfs:label "versionConcept"@en ;
    rdfs:range dblp:Publication .

dblp:versionInstance
    a rdf:Property ;
    rdfs:comment "The linked specific (instance) publication version."@en ;
    rdfs:domain dblp:VersionRelation ;
    rdfs:label "versionInstance"@en ;
    rdfs:range dblp:Publication .

dblp:versionLabel
    a rdf:Property ;
    rdfs:comment "The human-readable version label of the specific (instance) publication version."@en ;
    rdfs:domain dblp:VersionRelation ;
    rdfs:label "versionLabel"@en ;
    rdfs:range xsd:string .

dblp:versionOrdinal
    a rdf:Property ;
    rdfs:comment "The ordinal number of the specific (instance) publication version. This number is solely intended for sorting purposes: bigger numbers indicate later versions. Version ordinals do not need to describe a complete number range, nor is there a necessary relationship to the version labels."@en ;
    rdfs:domain dblp:VersionRelation ;
    rdfs:label "versionOrdinal"@en ;
    rdfs:range xsd:integer .

dblp:versionUri
    a rdf:Property ;
    rdfs:comment "An (optional) URI of identifying the linked specific (instance) publication version."@en ;
    rdfs:domain dblp:VersionRelation ;
    rdfs:label "versionUri"@en ;
    rdfs:range xsd:anyUri .

dblp:webpage
    a rdf:Property ;
    rdfs:comment "The URL of a web page about this item."@en ;
    rdfs:domain dblp:Entity ;
    rdfs:label "webpage"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf wd:P2699, schema:url ;
    owl:equivalentProperty foaf:page .

dblp:wikidata
    a rdf:Property ;
    rdfs:comment "A wikidata item."@en ;
    rdfs:domain dblp:Entity ;
    rdfs:label "wikidata"@en ;
    rdfs:range xsd:anyUri ;
    rdfs:subPropertyOf dblp:identifier ;
    owl:equivalentProperty locid:wikidata, datacite:wikidata .

dblp:wikipedia
    a rdf:Property ;
    rdfs:comment "The URL of an (English) Wikipedia article about this item."@en ;
    rdfs:domain dblp:Entity ;
    rdfs:label "wikipedia"@en ;
    rdfs:range foaf:Document ;
    rdfs:subPropertyOf dblp:webpage .

dblp:yearOfEvent
    a rdf:Property ;
    rdfs:comment "The year the conference or workshop contribution has been presented."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "yearOfEvent"@en ;
    rdfs:range xsd:gYear .

dblp:yearOfPublication
    a rdf:Property ;
    rdfs:comment "The year the publication's issue or volume has been published."@en ;
    rdfs:domain dblp:Publication ;
    rdfs:label "yearOfPublication"@en ;
    rdfs:range xsd:gYear ;
    owl:equivalentProperty dbo:publicationDate, dct:issued, wd:P577, schema:publication .
"""

# Input properties to find domain and range for
input_properties = [
    "https://dblp.org/rdf/schema#authoredBy",
    "https://dblp.org/rdf/schema#publishedIn"
]

# Initialize an RDF graph
g = Graph()

# Parse the ontology schema (assuming it's in Turtle format)
g.parse(data=ontology_schema, format="ttl")

# Define namespaces
DBLP = Namespace("https://dblp.org/rdf/schema#")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

# Dictionary to store the results
property_info = {}

# Function to get the subject (Domain) and object (Range) of a property
def get_domain_and_range(graph, prop_uri):
    """Fetches the domain and range for a given property URI from the graph."""
    prop = URIRef(prop_uri)
    
    # Get Domain (rdfs:domain)
    domains = list(graph.objects(prop, RDFS.domain))
    # Get Range (rdfs:range)
    ranges = list(graph.objects(prop, RDFS.range))
    
    # Simple logic: assume one domain and one range based on the DBLP schema structure
    domain = domains[0] if domains else None
    range_ = ranges[0] if ranges else None
    
    return domain, range_

# Process each input property
for prop_uri in input_properties:
    domain, range_ = get_domain_and_range(g, prop_uri)
    
    # Format the URIs to use the prefix 'dblp:' or 'xsd:' as in the output
    # This involves a simple string replacement based on the known namespaces
    def format_uri(uri):
        if not uri:
            return "None"
        uri_str = str(uri)
        if uri_str.startswith(str(DBLP)):
            return "dblp:" + uri_str.replace(str(DBLP), "")
        elif uri_str.startswith(str(XSD)):
            return "xsd:" + uri_str.replace(str(XSD), "")
        # Fallback for other namespaces if necessary, but DBLP properties in the prompt
        # only reference dblp: and xsd: types.
        return uri_str

    if domain and range_:
        formatted_domain = format_uri(domain)
        formatted_range = format_uri(range_)
        formatted_property = format_uri(URIRef(prop_uri))
        property_info[prop_uri] = f"<{formatted_domain}, {formatted_property}, {formatted_range}>"
    else:
        property_info[prop_uri] = f"<{format_uri(domain)}, {format_uri(URIRef(prop_uri))}, {format_uri(range_)}>"


# Generate the final output string in the specified format
output_list = [
    property_info.get("https://dblp.org/rdf/schema#authoredBy"),
    property_info.get("https://dblp.org/rdf/schema#publishedIn")
]
final_output = " ".join(output_list)

print(final_output)

<dblp:Publication, dblp:authoredBy, dblp:Creator> <dblp:Publication, dblp:publishedIn, xsd:string>


In [2]:
from rdflib import Graph, URIRef, RDFS, Namespace
import os

# Define the input properties
input_properties = [
    "https://dblp.org/rdf/schema#authoredBy",
    "https://dblp.org/rdf/schema#publishedIn"
]

# Define the file path for the ontology schema
ONTOLOGY_FILE_PATH = "/Users/sherrypan/GitHub/GAR_SKGQA/datasets/dblp/project_data/dblp_schema.ttl"

# Define namespaces for output formatting
DBLP = Namespace("https://dblp.org/rdf/schema#")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

def format_uri(uri):
    """Formats a full URI to its prefixed name (QName) or returns the string if not recognized."""
    if not uri:
        return "None"
    uri_str = str(uri)
    if uri_str.startswith(str(DBLP)):
        return "dblp:" + uri_str.replace(str(DBLP), "")
    elif uri_str.startswith(str(XSD)):
        return "xsd:" + uri_str.replace(str(XSD), "")
    return uri_str

def get_property_signature_from_file(file_path, properties):
    """
    Loads the ontology from a local file and extracts the (Domain, Property, Range) 
    signature for the given list of properties.
    """
    if not os.path.exists(file_path):
        return f"Error: File not found at {file_path}"
        
    g = Graph()
    
    try:
        # Load the graph directly from the Turtle file path
        print(f"Reading ontology from: {file_path}")
        g.parse(file_path, format="ttl")
    except Exception as e:
        return f"Error parsing ontology file: {e}"
    
    output_signatures = []

    for prop_uri in properties:
        prop = URIRef(prop_uri)
        
        # Query for rdfs:domain and rdfs:range
        # Assuming only one value for domain and range for simplicity of this task
        domains = list(g.objects(prop, RDFS.domain))
        ranges = list(g.objects(prop, RDFS.range))
        
        domain = domains[0] if domains else None
        range_ = ranges[0] if ranges else None
        
        formatted_domain = format_uri(domain)
        formatted_property = format_uri(prop)
        formatted_range = format_uri(range_)
        
        output_signatures.append(f"<{formatted_domain}, {formatted_property}, {formatted_range}>")
    
    return " ".join(output_signatures)

# Execute the function using the file path
result = get_property_signature_from_file(ONTOLOGY_FILE_PATH, input_properties)
print(result)

Reading ontology from: /Users/sherrypan/GitHub/GAR_SKGQA/datasets/dblp/project_data/dblp_schema.ttl
<dblp:Publication, dblp:authoredBy, dblp:Creator> <dblp:Publication, dblp:publishedIn, xsd:string>


In [5]:
import requests
from urllib.parse import quote_plus
from typing import List, Dict

# --- Configuration ---
# SPARQL_ENDPOINT = "https://sparql.dblp.org/sparql"
SPARQL_ENDPOINT = "http://localhost:7021"
INPUT_ENTITIES = [
    "<https://dblp.org/pid/82/9897>",
    "<https://dblp.org/pid/66/4077>"
]

# SPARQL query template (using format strings for insertion)
SPARQL_TEMPLATE = """
SELECT ?type
WHERE {{
  {entity_uri} <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> ?type .
}}
"""

# Define namespaces for abbreviation in output
DBLP_SCHEMA_NS = "https://dblp.org/rdf/schema#"

def format_uri(uri_str: str, namespace: str) -> str:
    """Abbreviates a full URI string using a provided namespace prefix (e.g., dblp:)."""
    if uri_str.startswith(namespace):
        return "dblp:" + uri_str[len(namespace):]
    return uri_str

def fetch_entity_types(endpoint: str, entities: List[str]) -> Dict[str, str]:
    """
    Executes SPARQL queries for a list of entities to find their RDF type.
    
    Args:
        endpoint: The SPARQL endpoint URL.
        entities: A list of entity URIs (as strings) to query.
        
    Returns:
        A dictionary mapping the original entity URI to its abbreviated type.
    """
    results_map = {}
    
    for entity_uri in entities:
        # 1. Format the SPARQL query
        # Remove the < > wrappers for interpolation into the query, but ensure the entity is still treated as a URI
        # The entity_uri variable already includes < and > as per the input format.
        query = SPARQL_TEMPLATE.format(entity_uri=entity_uri)
        
        # 2. Encode the query for the URL
        params = {'query': query, 'format': 'json'}
        
        # 3. Send the HTTP GET request
        try:
            response = requests.get(endpoint, params=params, timeout=10)
            response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
            data = response.json()

            # 4. Process the results
            types = data.get('results', {}).get('bindings', [])
            
            if types:
                # The query returns the type URI. We take the first one found.
                type_uri = types[0]['type']['value']
                
                # Format the entity URI for output (removing < >)
                clean_entity_uri = entity_uri.strip('<>')
                
                # Format the type URI (e.g., to dblp:Person)
                abbreviated_type = format_uri(type_uri, DBLP_SCHEMA_NS)
                
                results_map[clean_entity_uri] = abbreviated_type
            else:
                clean_entity_uri = entity_uri.strip('<>')
                results_map[clean_entity_uri] = "Unknown Type"

        except requests.exceptions.RequestException as e:
            print(f"Error querying {entity_uri}: {e}")
            results_map[entity_uri.strip('<>')] = "Query Failed"
            
    return results_map

def format_output(results: Dict[str, str]) -> str:
    """Formats the results dictionary into the requested string format."""
    output_parts = [f"<{entity}> : {entity_type}" for entity, entity_type in results.items()]
    return " ".join(output_parts)

# --- Execution ---
entity_types = fetch_entity_types(SPARQL_ENDPOINT, INPUT_ENTITIES)
final_output = format_output(entity_types)

print(final_output)

<https://dblp.org/pid/82/9897> : dblp:Creator <https://dblp.org/pid/66/4077> : dblp:Creator


In [6]:
a = "<https://dblp.org/rdf/schema#publishedIn>"

In [9]:
b = a.split("#")[-1].strip(">")

In [14]:
c = "dblp:" + b
print(c)  # Output: dblp:publishedIn

dblp:publishedIn


In [16]:
# read a csv file with three columns: property, domain and range
import pandas as pd

df = pd.read_csv("/Users/sherrypan/GitHub/GAR_SKGQA/datasets/dblp/project_data/dblp_property_schema.csv")
# given a list of properties, output their domain and range in the format <domain, property, range>
input_properties = [
    "https://dblp.org/rdf/schema#authoredBy",
    "https://dblp.org/rdf/schema#publishedIn"
]
output_list = []
for prop in input_properties:
    prop_name = "dblp:" + prop.split("#")[-1].strip(">")
    row = df[df['property'] == prop_name]
    if not row.empty:
        domain = row['domain'].values[0]
        range_ = row['range'].values[0]
        output_list.append(f"<{domain}, {prop_name}, {range_}>")



In [17]:
output_list

['<dblp:Publication, dblp:authoredBy, dblp:Creator>',
 '<dblp:Publication, dblp:publishedIn, xsd:string>']

In [22]:

entities = str("<https://dblp.org/pid/229/5178>, <https://dblp.org/pid/73/8951>").split(", ")
endpoint_url = "http://localhost:7021/sparql"
from SPARQLWrapper import SPARQLWrapper, JSON, POST
SPARQL_TEMPLATE = """
    SELECT ?type
    WHERE {{
    {entity_uri} <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> ?type .
    }}
    """
entity_types_list = []
for entity_uri in entities:
    query = SPARQL_TEMPLATE.format(entity_uri=entity_uri)
    wrapper = SPARQLWrapper(endpoint_url)
    wrapper.setMethod(POST)
    wrapper.setReturnFormat(JSON)
    wrapper.setQuery(query)

    results = wrapper.query().convert()
    types = [result["type"]["value"] for result in results["results"]["bindings"]]
    # Format the output
    # Abbreviate the type URIs to use the 'dblp:' prefix where applicable
    abbreviated_types = []
    for t in types:
        if t.startswith("https://dblp.org/rdf/schema#"):
            abbreviated_types.append("dblp:" + t.split("#")[-1])
        else:
            abbreviated_types.append(t)
    types = abbreviated_types
    entity_types = f"{entity_uri}: " + ", ".join(types)
    entity_types_list.append(entity_types)
final_output = " \n".join(entity_types_list)
print(final_output)

<https://dblp.org/pid/229/5178>: dblp:Creator, dblp:Person 
<https://dblp.org/pid/73/8951>: dblp:Creator, dblp:Person


In [24]:
file = "/Users/sherrypan/GitHub/GAR_SKGQA/results/dblp/zero_triple_questions.csv"

zero_triple_list = []
df = pd.read_csv(file)
for _, row in df.iterrows():
    if row['triples'] == 0:
        zero_triple_list.append(row['id'])

In [30]:
# read a csv with question id
file = "/Users/sherrypan/GitHub/GAR_SKGQA/results/dblp/non_zero_ids.csv"
non_zero_ids = []
df = pd.read_csv(file)
for _, row in df.iterrows():
    non_zero_ids.append(row['id'])
zero_triple_list = []
for i in range(1, 2001):
    id_str = f"Q{i:04d}"
    if id_str not in non_zero_ids:
        zero_triple_list.append(id_str)
print(zero_triple_list)  # Output the list of question ids with zero triples
# save the list to a csv file
output_file = "/Users/sherrypan/GitHub/GAR_SKGQA/results/dblp/zero_triple_questions.csv"
df = pd.DataFrame(zero_triple_list, columns=['id'])
df.to_csv(output_file, index=False)

['Q0319', 'Q0464', 'Q0565', 'Q0591', 'Q0607', 'Q0614', 'Q0617', 'Q0621', 'Q0634', 'Q0649', 'Q0653', 'Q0655', 'Q0658', 'Q0667', 'Q0674', 'Q0687', 'Q0703', 'Q0764', 'Q0814', 'Q0830', 'Q0961', 'Q0984', 'Q1103', 'Q1111', 'Q1159', 'Q1226', 'Q1264', 'Q1266', 'Q1297', 'Q1301', 'Q1302', 'Q1303', 'Q1305', 'Q1310', 'Q1315', 'Q1316', 'Q1319', 'Q1322', 'Q1326', 'Q1331', 'Q1334', 'Q1335', 'Q1336', 'Q1337', 'Q1338', 'Q1340', 'Q1341', 'Q1343', 'Q1345', 'Q1347', 'Q1350', 'Q1351', 'Q1352', 'Q1353', 'Q1355', 'Q1358', 'Q1362', 'Q1365', 'Q1367', 'Q1372', 'Q1376', 'Q1381', 'Q1384', 'Q1387', 'Q1388', 'Q1390', 'Q1391', 'Q1393', 'Q1394', 'Q1395', 'Q1396', 'Q1397', 'Q1398', 'Q1400', 'Q1406', 'Q1415', 'Q1416', 'Q1421', 'Q1445', 'Q1454', 'Q1456', 'Q1458', 'Q1488', 'Q1489', 'Q1505', 'Q1516', 'Q1521', 'Q1525', 'Q1533', 'Q1534', 'Q1541', 'Q1548', 'Q1549', 'Q1550', 'Q1555', 'Q1558', 'Q1562', 'Q1574', 'Q1590', 'Q1595', 'Q1772', 'Q1780', 'Q1781', 'Q1782', 'Q1787', 'Q1788', 'Q1789', 'Q1794', 'Q1795', 'Q1797', 'Q1802', 

In [46]:
query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
select *
WHERE {
<https://dblp.org/rec/conf/acl/CirikMB22> dblp:authoredBy ?author .
}
limit 1
"""
wrapper = SPARQLWrapper("https://sparql.dblp.org/sparql")
# local dblp SPARQL endpoint: http://localhost:7020
wrapper.setMethod(POST)
wrapper.setReturnFormat(JSON)
wrapper.setQuery(query)
results = wrapper.query().convert()
print(results)

{'head': {'vars': ['author']}, 'results': {'bindings': [{'author': {'type': 'uri', 'value': 'https://dblp.org/pid/136/8682'}}]}}
